importation des bibliotech 

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


chargement de donne clean

In [19]:
df = pd.read_csv(
    "../data/processed/clean_data.csv"
)


### Definir les valeur de entrainment et valuer de test 

In [ ]:
input = df.drop("selling_price",axis=1)
output = df["selling_price"]

# separation de dataset
X_train, X_test, y_train, y_test = train_test_split(
    input,output,test_size=0.2,random_state=42
)


### Cree la pipline de entrainement 

In [ ]:
numeric = ["year","km_driven"]
categorical = [
    "fuel",
    "seller_type",
    "transmission",
    "owner"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),numeric),
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical)
    ]
)
preprocessor.fit_transform(X_train)



### linear Regression

In [45]:

model_linear = Pipeline([
    ("preprocessor",preprocessor),
    ("model",LinearRegression())
])

model_linear.fit(X_train,y_train)

y_predict_linear = model_linear.predict(X_test)


# test 
mae_linear = mean_absolute_error(y_test,y_predict_linear)
rmse_linear = np.sqrt(mean_squared_error(y_test,y_predict_linear))
r2_linear = r2_score(y_test,y_predict_linear)

print("MAE : ",mae_linear)
print("RMSE : ",rmse_linear)
print("R2 : ",r2_linear)

MAE :  202515.20184031726
RMSE :  360364.82441312267
R2 :  0.42562314048713723


### Random forest

In [48]:
model_random_1 = Pipeline([
    ("preprocessor",preprocessor),
    ("model",RandomForestRegressor())
])

model_random_1.fit(X_train,y_train)

y_predict_random_1 = model_random_1.predict(X_test)

# test 
mae_random_1 = mean_absolute_error(y_test,y_predict_random_1)
rmse_random_1 = np.sqrt(mean_squared_error(y_test,y_predict_random_1))
r2_random_1 = r2_score(y_test,y_predict_random_1)

print("MAE : ",mae_random_1)
print("RMSE : ",rmse_random_1)
print("R2 : ",r2_random_1)

MAE :  188500.83064988244
RMSE :  371453.9192785589
R2 :  0.3897299736746741


### optimisation de random forest

In [ ]:
param_grid_random = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5, 10]
}

grid_random = GridSearchCV(
    estimator=model_random_1,
    param_grid=param_grid_random,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

grid_random.fit(X_train, y_train)

print(grid_random.best_params_)


{'model__max_depth': 5, 'model__min_samples_split': 10, 'model__n_estimators': 300}


### apres optimisation

In [50]:
model_random = Pipeline([
    ("preprocessor",preprocessor),
    ("model",RandomForestRegressor(
        max_depth = 5,
        min_samples_split = 10,
        n_estimators = 300
    ))
])

model_random.fit(X_train,y_train)

y_predict_random = model_random.predict(X_test)

# test 
mae_random = mean_absolute_error(y_test,y_predict_random)
rmse_random = np.sqrt(mean_squared_error(y_test,y_predict_random))
r2_random = r2_score(y_test,y_predict_random)

print("MAE : ",mae_random)
print("RMSE : ",rmse_random)
print("R2 : ",r2_random)

MAE :  174442.84723119254
RMSE :  333012.1462232359
R2 :  0.5095075823027745


### xGboost

In [53]:
model_xgboost_1= Pipeline([
    ("preprocessor",preprocessor),
    ("model",XGBRegressor())
])

model_xgboost_1.fit(X_train,y_train)

y_predict_xgboost_1 = model_xgboost_1.predict(X_test)

# test 
mae_xgboost_1 = mean_absolute_error(y_test,y_predict_xgboost_1)
rmse_xgboost_1 = np.sqrt(mean_squared_error(y_test,y_predict_xgboost_1))
r2_xgboost_1 = r2_score(y_test,y_predict_xgboost_1)

print("MAE : ",mae_xgboost_1)
print("RMSE : ",rmse_xgboost_1)
print("R2 : ",r2_xgboost_1)

MAE :  196679.953125
RMSE :  424178.44275257556
R2 :  0.20419007539749146


### test pour les melleur parametre 

In [ ]:
param_grid = {
    "model__learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3],
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 5, 7]
}

grid_search = GridSearchCV(
    estimator=model_xgboost_1,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

grid_search.fit(X_train,y_train)

print(grid_search.best_params_)


{'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 300}


apres l'optimisation


In [52]:
model_xgboost= Pipeline([
    ("preprocessor",preprocessor),
    ("model",XGBRegressor(
        learning_rate = 0.01,
        n_estimators = 300,
        max_depth = 3
    ))
])

model_xgboost.fit(X_train,y_train)

y_predict_xgboost = model_xgboost.predict(X_test)

# test 
mae_xgboost = mean_absolute_error(y_test,y_predict_xgboost)
rmse_xgboost = np.sqrt(mean_squared_error(y_test,y_predict_xgboost))
r2_xgboost = r2_score(y_test,y_predict_xgboost)

print("MAE : ",mae_xgboost)
print("RMSE : ",rmse_xgboost)
print("R2 : ",r2_xgboost)

MAE :  175894.46875
RMSE :  328765.8757961355
R2 :  0.5219364166259766


### SVR  Support Vector Regression

In [27]:
model_svr = Pipeline([
    ("preprocessor",preprocessor),
    ("model",SVR())
])
model_svr.fit(X_train, y_train)

y_pred_svr = model_svr.predict(X_test)

# test 
mae_svr = mean_absolute_error(y_test,y_pred_svr)
rmse_svr = np.sqrt(mean_squared_error(y_test,y_pred_svr))
r2_svr = r2_score(y_test,y_pred_svr)

print("MAE : ",mae_svr)
print("RMSE : ",rmse_svr)
print("R2 : ",r2_svr)

MAE :  263554.00334811205
RMSE :  488411.67044208315
R2 :  -0.05507714458107693


In [54]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Random Forest Optimise",
        "XGBoost",
        "XGBoost optimise",
        "SVR"
    ],
    "MAE": [
        mae_linear,
        mae_random_1,
        mae_random,
        mae_xgboost_1,
        mae_xgboost,
        mae_svr
    ],
    "RMSE": [
        rmse_linear,
        rmse_random_1,
        rmse_random,
        rmse_xgboost_1,
        rmse_xgboost,
        rmse_svr
    ],
    "R2": [
        r2_linear,
        r2_random_1,
        r2_random,
        r2_xgboost_1,
        r2_xgboost,
        r2_svr
    ]
}).round(2)

results.to_csv(
    "../data/processed/baseline_results.csv",
    index=False
)

In [44]:
nouvelle_voiture = pd.DataFrame({
    "name": ["Maruti Swift Dzire VDI"],
    "year": [2018],
    "km_driven": [50000],
    "fuel": ["Diesel"],
    "seller_type": ["Individual"],
    "transmission": ["Manual"],
    "owner": ["First Owner"]
})

prediction_xgboost = model_xgboost.predict(nouvelle_voiture)
prediction_linear = model_linear.predict(nouvelle_voiture)
prediction_random = model_random.predict(nouvelle_voiture)

print(f"Prix prédit xgboost : {prediction_xgboost[0]:.2f}")
print(f"Prix prédit linear regression : {prediction_linear[0]:.2f}")
print(f"Prix prédit randon forest : {prediction_random[0]:.2f}")

Prix prédit xgboost : 845656.00
Prix prédit linear regression : 717620.98
Prix prédit randon forest : 670581.11
